<a href="https://colab.research.google.com/github/david91107/DataSciencePortfolio/blob/main/pytorch-projects/notebook/NLP_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overview

This is a project to learn how to build a model using transformers.In the context of artificial intelligence, a transformer is a type of deep learning model that has revolutionized the field of natural language processing (NLP) and beyond. Introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017, transformers are designed to handle sequential data, like text, in a more efficient and effective way compared to previous models like recurrent neural networks (RNNs) and long short-term memory networks (LSTMs).

Transformers are characterized by their use of self-attention mechanisms, which allow them to weigh the importance of different parts of the input data differently. This is particularly useful in understanding the context and relationships between words in a sentence or elements in a sequence, without being constrained by the sequence's order or proximity.

Key features and advantages of transformers include:

Parallelization: Unlike RNNs and LSTMs, which process data sequentially, transformers can process entire sequences of data in parallel. This significantly reduces training times and allows for more efficient computation.
Scalability: Transformers can be scaled up with more layers and parameters to handle larger datasets and more complex tasks, making them highly effective for a wide range of applications.
Flexibility: They can be adapted for a variety of tasks beyond text processing, including image recognition, audio processing, and even tasks in other domains like genomics.
Self-Attention: The self-attention mechanism allows transformers to consider the entire context of a sequence when processing each element, leading to a better understanding of the data.
Transfer Learning: Models based on transformers, such as BERT (Bidirectional Encoder Representations from Transformers), GPT (Generative Pretrained Transformer), and others, have shown remarkable performance in transfer learning, where a model trained on one task can be adapted for another related task with minimal additional training.
Transformers are now a foundational element in the development of state-of-the-art AI systems for natural language understanding, text generation, translation, and more, pushing the boundaries of what's possible in AI research and applications.


Ill be creating a corsus from a classic car forum powered by https://xenforo.com/. The intent is to goal is to create a "virtual mechanic" to help maintain older cars that have a dwindling set of experts available to turn to.

Raw text vs pre-processed

1. Fine-Tuning BERT or Similar Models
<br>
Pros:<br>
Effective with Preprocessed and Raw Text: Can handle both preprocessed (for cleaner, more focused analysis) and raw text (to capture nuances and specific jargon).
Good with Varied Text Lengths: Capable of managing both short and long texts, though long texts might need to be segmented due to token limits.
<br>
Cons:<br>
Token Limitations for Long Texts: BERT has a token limit (512 tokens), necessitating additional processing for longer texts.
Resource-Intensive: Requires significant computational resources, especially for processing and fine-tuning on large datasets.
<br>
<br>
2. Using GPT-3 for Response Generation
<br>
Pros:<br>
Adaptable to Both Text Types: Can generate responses based on both preprocessed and raw text, utilizing the context effectively.
Handles Long Text Well: GPT-3's architecture allows it to handle longer sequences of text better than many other models, making it suitable for detailed responses.
<br>
Cons:<br>
Context Dilution in Long Texts: Might lose context in very long text sequences, affecting response relevance.
Operational Cost and Accessibility: Using GPT-3, particularly at scale, can be expensive and subject to usage constraints.
<br>
<br>
3. Hybrid Approach (Combining Fine-Tuned BERT and GPT-3)
<br>
Pros:<br>
Versatility with Text Types: Combines BERT's analytical strength (which can handle both preprocessed and raw text) with GPT-3’s generative capabilities.
Effective with All Text Lengths: Can manage short texts efficiently and long texts through a combination of BERT's analytical precision and GPT-3's generative scope.
<br>
Cons:<br>
Integration Complexity: Managing and integrating two systems, especially ensuring coherence between them, adds complexity.
High Resource Requirement: Both models are resource-intensive, which could be a significant factor, particularly for processing large datasets.

### Create Enviornment

In [1]:
!pip3 install pandas
!pip3 install numpy
!pip3 install requests
!pip3 install beautifulsoup4
!pip3 install transformers
!pip3 install torch


import pandas as pd

import numpy as np

import requests

from bs4 import BeautifulSoup

import re

import nltk
#nltk.download("all")
#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')
#nltk.download('vader_lexicon')

# from nltk.tokenize import word_tokenize
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer
# from nltk.tag import pos_tag

from transformers import pipeline

In [2]:
# This seems to propagate credentials better from its own cell

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Database Schema

Following is my initial design for the final database schema.


Table 1: Issues
<br>
*   Key Should be from pandas
*   Issue ID (Foreign Key) Should be the thread_id
*   Issue Should be the thread title
*   Short Description Should be from the thread post
*   Keywords: Should be from the thread post

Table 2: Solutions
<br>
*   Key Should be from pandas
*   Issue ID (Foreign Key) Should be the thread_id
*   Detailed Solution Should be from the thread posts
*   Keywords Should be from the thread posts

Table 3: Notes
<br>
*   Key Should be from pandas
*   Issue ID (Foreign Key) Should be taken from the thread_key
*   Unstructured Note Content Should be from the thread posts

## Create corpus

Ill be scraping post from my classic car forum for the corpus. Ill be limiting the data retreival while I build the model so I dont impact the site for users. Ill be using Beautiful Soup where possible to parse the content into a dataframe structure.

In [3]:
# Create list of URLs to scrape and parse

import pandas as pd

threads = 100

import pandas as pd

data = []

def create_urls(threads, page_number=1):
    base_url = 'https://e9coupe.com/forum/threads/'
    # Iterate over thread IDs to generate URLs
    for thread_id in range(1, threads + 1):
        thread_url = f"{base_url}{thread_id}"
        data.append({'thread_id': thread_id, 'thread_url': thread_url})

# Call the function with the correct parameter
create_urls(threads)  # Using the 'threads' variable

# Convert the list of dictionaries into a DataFrame
df_urls = pd.DataFrame(data)

# Display the resulting DataFrame
df_urls.head()

,thread_id,thread_url
0,1,https://e9coupe.com/forum/threads/1
1,2,https://e9coupe.com/forum/threads/2
2,3,https://e9coupe.com/forum/threads/3
3,4,https://e9coupe.com/forum/threads/4
4,5,https://e9coupe.com/forum/threads/5


In [4]:
# Process URLs into a dataframe of thread ids and thread titles. This will be
# the core table while I decorate the dataframe with additional metadata.

pages = 1

data = []

df_threads = pd.DataFrame()

def fetch_thread_data(df):
    for index, row in df.iterrows():
        thread_id = row['thread_id']
        url = row['thread_url']
        for i in range(1, pages + 1):
            page_url = f"{url}/?page={i}"  # Construct the page URL
            response = requests.get(page_url)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                title = soup.find('title').get_text()
                thread_title = title.split('|')[0].strip()  # Extract the thread title

                data.append({'thread_id': thread_id, 'thread_title': thread_title, 'thread_url': page_url})

    return data

# Fetch thread data
data = fetch_thread_data(df_urls)

# Convert the list of dictionaries into a DataFrame
df_threads = pd.DataFrame(data)

# Display the resulting DataFrame
df_threads.head()

,thread_id,thread_title,thread_url
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1


In [5]:
# Extract the inital content when the thread was created.
# This text will be used to create a short description.

def fetch_first_post_content(df):
    data = []

    for thread_id, url in zip(df['thread_id'], df['thread_url']): # ensures pairing
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        first_post = soup.find('article', class_='message-body')
        post_content = first_post.get_text(strip=True)

        data.append({'thread_id': thread_id, 'thread_first_post': post_content})

    return data

# Fetch first post content and convert to DataFrame
data = fetch_first_post_content(df_threads)
first_post_df = pd.DataFrame(data)

# Casting the values I want to join on
df_threads['thread_id'] = df_threads['thread_id'].astype(int)
first_post_df['thread_id'] = first_post_df['thread_id'].astype(int)

# Update the df
df_threads = pd.merge(df_threads, first_post_df, on='thread_id', how='left')

# Display the resulting DataFrame
df_threads.head()

,thread_id,thread_title,thread_url,thread_first_post
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam..."
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...


### Summarization
Very difficult to find a winning strategy here that can accomidate both long and short length text blocks.

**Extractive Summarization**
<br>
Pros:
<br>
Good with Raw Text: Extractive methods can work directly with raw, unstructured text, as they mainly focus on selecting key sentences or phrases without needing deep linguistic processing.
Straightforward Implementation: These methods do not require complex preprocessing like tokenization or lemmatization, simplifying their implementation.
<br>
Cons:
<br>
Limited Depth in Understanding: While they can handle raw text, they may not fully capture the nuanced meaning, especially when the text contains complex structures or unorthodox language use.
Less Effective with Poorly Structured Text: In cases where the text is poorly structured or highly informal, extractive summarization might struggle to identify the main points effectively.
<br>
<br>


**Abstractive Summarization** (like sshleifer/distilbart-cnn-12-6)
<br>
Pros:
<br>
Advanced Processing Capabilities: Abstractive models, especially those based on transformer architectures, are designed to handle and interpret raw text, capturing deeper linguistic and contextual nuances.
Higher Tolerance for Unstructured Text: These models can manage unstructured or informal text by understanding and then rephrasing it in a more coherent and structured summary.
<br>
Cons:
<br>
Dependence on Preprocessing for Optimal Performance: While they can process raw text, the quality of the output can be significantly improved with proper tokenization and lemmatization, especially for complex texts.
Potential Overhead: Requires more computational resources to process and understand raw text, which might be more efficiently handled with some level of preprocessing.
<br>
<br>
**Hybrid Summarization**
<br>
Pros:
<br>
Flexibility in Text Processing: Combining extractive and abstractive methods allows for handling both raw and preprocessed text, adapting to the text's structure and complexity.
Balanced Approach: Can leverage the strengths of extractive methods in handling raw text for identifying key points, while using abstractive techniques for generating a coherent summary.
<br>
Cons:
<br>
Complex Preprocessing Requirements: The need to integrate both extractive and abstractive approaches may necessitate more sophisticated preprocessing strategies to optimize performance.
Potential for Processing Inefficiencies: The combined approach might lead to redundancies or inefficiencies in processing, especially if the text is either too raw or overly preprocessed.
<br>
<br>
After attempting sshleifer/distilbart-cnn-12-6 I found it had a character limit (1024) that is too restrictive for my needs. However, T5 has no limits and is what Ill be trying.

In [6]:
# This is a summarization of the initial thread post

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Initialize the T5 tokenizer and model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

def summarize_text(df):

    thread_first_post_summary = []  # Initialize the list to hold summaries
    for thread_id, text in zip(df['thread_id'], df['thread_first_post']): # ensures pairing

        # Prefixing the input text with "summarize: " as T5 expects
        inputs = tokenizer.encode("summarize: " + text, return_tensors="pt", truncation=True, max_length=512)
        summary_ids = model.generate(inputs, max_length=150, length_penalty=2.0, num_beams=4, early_stopping=True)

        # Decode the generated ids to get the summary text
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        # Append the tuple containing thread_id and summary to the list
        thread_first_post_summary.append((thread_id, summary))

    return thread_first_post_summary

# Fetch first post content and convert to DataFrame
data = summarize_text(df_threads)
df_short_desc = pd.DataFrame(data, columns=['thread_id', 'thread_first_post_summary'])

# Merge the summarized_df with df_threads on the 'thread_id' column
df_threads = pd.merge(df_threads, df_short_desc, on='thread_id', how='left')

# Display the resulting DataFrame
df_threads.head()

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,thread_id,thread_title,thread_url,thread_first_post,thread_first_post_summary
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...,goin down to NC to drive her home to new cs. I...
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam...",new beige carpet kit from Seams Perfect availa...
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...,"the board went MIA for a few days, and mye28.c..."
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...,the current is a Getrag 225 and the speedo cab...
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...


### Keywords

In [7]:
# Gather keywords from the initial thread post
# Keyword extraction with BERT
from transformers import BertTokenizer, BertModel

import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to extract keywords using BERT
def bert_extract_keywords(text, tokenizer, model, top_n=5):
    # Tokenize and encode the text
    inputs = tokenizer.encode_plus(text, add_special_tokens=True, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs['input_ids'][0]

    # Get the embeddings from the last hidden layer
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.squeeze(0)

    # Compute word importance by summing up the embeddings
    word_importance = torch.sum(embeddings, dim=1)

    # Get the indices of the top n important words
    top_n_indices = word_importance.argsort(descending=True)[:top_n]

    # Filter out indices that are out of range of input_ids
    top_n_indices = [idx for idx in top_n_indices if idx < len(input_ids)]

    # Decode the top n words
    keywords = [tokenizer.decode([input_ids[idx]]) for idx in top_n_indices]

    return keywords

# df_threads['keywords'] = bert_extract_keywords(df_threads['summary'],tokenizer, model)
df_threads['thread_first_post_keywords'] = df_threads['thread_first_post_summary'].apply(lambda x: bert_extract_keywords(x, tokenizer, model))

# Display the resulting DataFrame
df_threads.head()


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

,thread_id,thread_title,thread_url,thread_first_post,thread_first_post_summary,thread_first_post_keywords
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...,goin down to NC to drive her home to new cs. I...,"[to, new, status, home, [CLS]]"
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam...",new beige carpet kit from Seams Perfect availa...,"[new, beige, 100, kit, from]"
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...,"the board went MIA for a few days, and mye28.c...","[into, carving, up, [CLS], the]"
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...,the current is a Getrag 225 and the speedo cab...,"[not, 225, soft, cable, a]"
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...,"[##l, super, listed, a, the]"


In [8]:
# Process threads and fetch all post data'

# As written this will fetch all the posts on the first page, which is 20
# This might need to be updated to iterate through all page values (1 through n)


def fetch_and_parse_thread(df):
    post_data = []
    processed_posts = set()
    for index, row in df.iterrows():
        response = requests.get(row['thread_url'])
        soup = BeautifulSoup(response.text, 'html.parser')
        articles = soup.find_all('article', class_='message--post')  # Correct class name as example
        for article in articles:
            post_id = article.get('id', 'N/A')
            numeric_post_id = re.findall(r'\d+', post_id)[0] if re.findall(r'\d+', post_id) else 'N/A'

            if numeric_post_id not in processed_posts:
                processed_posts.add(numeric_post_id)
                content = article.find('div', class_='bbWrapper').get_text(strip=True)
                #timestamp = article.find('time', class_='u-dt').get_text(strip=True) if article.find('time', class_='u-dt') else 'N/A'
                #post_number_element = article.find('ul', class_='message-attribution-opposite').find('li').find_next_sibling('li')
                #post_number = post_number_element.get_text(strip=True) if post_number_element else 'N/A'
                #post_number = post_number.lstrip('#') if post_number != 'N/A' else post_number

                post_data.append({
                    'thread_id': row['thread_id'],  # Corrected to use row's data
                    'post_id': numeric_post_id,
                    #'post_number': post_number,
                    'post_raw': content
                })

    return pd.DataFrame(post_data, columns=['thread_id', 'post_id','post_raw'])

# Fetch thread URLs and titles, and store in a DataFrame
df_posts = fetch_and_parse_thread(df_threads)

# Display the resulting DataFrame
df_posts.head()

,thread_id,post_id,post_raw
0,1,1,New cs owner here! Goin down to NC to drive h...
1,1,2,Congrats! Where in NJ are you?
2,1,4,New OwnerCongrats pecsok....from one new owner...
3,1,6,"Hi Chuck,How did things go? I imagine the car ..."
4,1,16,"Agave CoupeHi Len,Car is on the way, looks gre..."


In [9]:
# This is a summarization of all posts for a given thread

# Count the number of unique threads to ensure I dont drop any
print(df_posts['thread_id'].nunique())

aggregated_posts = df_posts.groupby(['thread_id'])['post_raw'].apply(lambda x: ' '.join(x)).reset_index(name='post_concat')

df_threads = df_threads.merge(aggregated_posts, on=['thread_id'], how='left')

# Display the resulting DataFrame
df_threads.head()

100


,thread_id,thread_title,thread_url,thread_first_post,thread_first_post_summary,thread_first_post_keywords,post_concat
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...,goin down to NC to drive her home to new cs. I...,"[to, new, status, home, [CLS]]",New cs owner here! Goin down to NC to drive h...
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam...",new beige carpet kit from Seams Perfect availa...,"[new, beige, 100, kit, from]","Hello all,Brand new beige carpet kit from Seam..."
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...,"the board went MIA for a few days, and mye28.c...","[into, carving, up, [CLS], the]",And it looksfamiliar...I suspected this might ...
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...,the current is a Getrag 225 and the speedo cab...,"[not, 225, soft, cable, a]",I have just purchased a CS and discovered that...
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...,"[##l, super, listed, a, the]",My spring itch is back and i'm looking for the...


### Pre-process the text

### Short Description

In [10]:
# This is a summarization of all posts for a given thread

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Initialize the T5 tokenizer and model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

def summarize_text(df):
    sum_text = []  # Initialize the list to hold summaries
    for text in df['post_concat']:
        # Ensure the text is a string and not empty
        #if not isinstance(text, str) or not text.strip():
        #    sum_text.append("")  # Append an empty string for non-valid entries
        #   continue

        # Prefixing the input text with "summarize: " as T5 expects
        inputs = tokenizer.encode("summarize: " + text, return_tensors="pt", truncation=True, max_length=512)
        summary_ids = model.generate(inputs, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)

        # Decode the generated ids to get the summary text
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        sum_text.append(summary)

    return sum_text

df_threads['post_summary'] = summarize_text(df_threads)

# Display the resulting DataFrame
df_threads.head()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


,thread_id,thread_title,thread_url,thread_first_post,thread_first_post_summary,thread_first_post_keywords,post_concat,post_summary
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...,goin down to NC to drive her home to new cs. I...,"[to, new, status, home, [CLS]]",New cs owner here! Goin down to NC to drive h...,cs owner goin down to NC to drive her home to ...
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam...",new beige carpet kit from Seams Perfect availa...,"[new, beige, 100, kit, from]","Hello all,Brand new beige carpet kit from Seam...",beige carpet kit from Seams Perfect is $100 be...
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...,"the board went MIA for a few days, and mye28.c...","[into, carving, up, [CLS], the]",And it looksfamiliar...I suspected this might ...,"DanG, will we be carving up the board into for..."
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...,the current is a Getrag 225 and the speedo cab...,"[not, 225, soft, cable, a]",I have just purchased a CS and discovered that...,"doug TrannyHi Doug,You want a 265 Getrag for y..."
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...,"[##l, super, listed, a, the]",My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...


In [11]:
from transformers import BertTokenizer, BertModel

import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to extract keywords using BERT
def bert_extract_keywords(text, tokenizer, model, top_n=5):
    # Tokenize and encode the text
    inputs = tokenizer.encode_plus(text, add_special_tokens=True, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs['input_ids'][0]

    # Get the embeddings from the last hidden layer
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.squeeze(0)

    # Compute word importance by summing up the embeddings
    word_importance = torch.sum(embeddings, dim=1)

    # Get the indices of the top n important words
    top_n_indices = word_importance.argsort(descending=True)[:top_n]

    # Filter out indices that are out of range of input_ids
    top_n_indices = [idx for idx in top_n_indices if idx < len(input_ids)]

    # Decode the top n words
    keywords = [tokenizer.decode([input_ids[idx]]) for idx in top_n_indices]

    return keywords

df_threads['post_keywords'] = df_threads['post_summary'].apply(lambda x: bert_extract_keywords(x, tokenizer, model))

# Display the resulting DataFrame
df_threads.head()

,thread_id,thread_title,thread_url,thread_first_post,thread_first_post_summary,thread_first_post_keywords,post_concat,post_summary,post_keywords
0,1,Welcome all!,https://e9coupe.com/forum/threads/1/?page=1,New cs owner here! Goin down to NC to drive h...,goin down to NC to drive her home to new cs. I...,"[to, new, status, home, [CLS]]",New cs owner here! Goin down to NC to drive h...,cs owner goin down to NC to drive her home to ...,"[owner, to, status, to, post]"
1,2,Brand New Beige Carpet Kit Available,https://e9coupe.com/forum/threads/2/?page=1,"Hello all,Brand new beige carpet kit from Seam...",new beige carpet kit from Seams Perfect availa...,"[new, beige, 100, kit, from]","Hello all,Brand new beige carpet kit from Seam...",beige carpet kit from Seams Perfect is $100 be...,"[beige, in, kit, ready, install]"
2,3,It's alive!,https://e9coupe.com/forum/threads/3/?page=1,And it looksfamiliar...I suspected this might ...,"the board went MIA for a few days, and mye28.c...","[into, carving, up, [CLS], the]",And it looksfamiliar...I suspected this might ...,"DanG, will we be carving up the board into for...","[into, carving, up, ,, ,]"
3,4,5 Speed Tranny,https://e9coupe.com/forum/threads/4/?page=1,I have just purchased a CS and discovered that...,the current is a Getrag 225 and the speedo cab...,"[not, 225, soft, cable, a]",I have just purchased a CS and discovered that...,"doug TrannyHi Doug,You want a 265 Getrag for y...","[##hi, ., for, [CLS], if]"
4,5,Restored or Original?,https://e9coupe.com/forum/threads/5/?page=1,My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...,"[##l, super, listed, a, the]",My spring itch is back and i'm looking for the...,i've been considering the super low mileage CS...,"[##l, super, listed, with, a]"


In [12]:
# Export and save result
df_threads.to_csv('/content/drive/MyDrive/e9/nlp/df_threads.csv', index=False)

####Table 1: Issues

*   ID (Unique ID)
*   Issue
*   Short Description
*   Keywords




In [19]:
#Table 1 Issues

#*   Key Should be from pandas
#*   Issue ID (Foreign Key) Should be taken from the thread_id
#*   Issue Should be the thread title
#*   Short Description Using the thread title for now. Should be the post of the originating thread
#*   Keywords: Should be from the thread post

# Table 1
df_table_1 = df_threads[['thread_id','thread_title','thread_first_post_summary','thread_first_post_keywords']]

# Export and save result
df_table_1.to_csv('/content/drive/MyDrive/e9/nlp/df_e9_table_1.csv', index=False)


####Table 2: Solutions

*   ID (Unique Note ID)
*   Issue
*   Issue ID (Foreign Key linking to ID in Issues table)
*   Solution


In [14]:
#Table 2
#*   Key Should be from pandas
#*   Issue Should be the thread title
#*   Issue ID (Foreign Key) Should be taken from the thread_key
#*   Detailed Solution Should be the concatinated post_raw per thread_id
#*   Keywords Should be from be the concatinated post_raw per thread_id

df_table_2 = df_threads[['thread_title','thread_id','post_concat','post_keywords']]

# Export and save result
df_table_2.to_csv('/content/drive/MyDrive/e9/nlp/df_e9_table_2.csv', index=False)

####Table 3: Notes

Key Should be from pandas
Issue ID (Foreign Key) Should be taken from the thread_key
Unstructured Note Content Should be from the thread posts

In [15]:
# Table 3: Notes

#*   Key Should be from pandas
#*   Issue ID (Foreign Key) Should be taken from the thread_key
#*   Unstructured Content: Should be from the raw thread posts

df_table_3 = df_threads[['thread_id','post_concat']]

# Export and save result
df_table_3.to_csv('/content/drive/MyDrive/e9/nlp/df_e9_table_3.csv', index=False)

What if I processed a complete auto repair guide to train on ??

ChatGPT
Training a keyword extraction model on a complete auto repair guide or similar domain-specific text is an excellent idea, especially if your goal is to improve the relevance and accuracy of keyword extraction for automotive discussions. Here's how you can approach this:

Domain-Specific Corpus: Obtain a corpus of domain-specific texts, such as auto repair manuals, guides, or technical documents. This corpus should contain text that is closely related to the automotive domain and covers various aspects of auto repair, maintenance, and troubleshooting.

Preprocessing: Preprocess the text data, similar to what you did with your forum data. This includes tokenization, stop word removal, lemmatization, and any other necessary cleaning steps.

Keyword Labeling: Manually label or annotate the relevant keywords in this domain-specific text. You can do this by identifying words or phrases that are highly relevant to specific topics, parts, or procedures in the auto repair domain. This labeled data will serve as your training dataset.

Training a Keyword Extraction Model: Utilize a machine learning or NLP model to train on this labeled dataset. You can explore different models, including traditional machine learning models (e.g., SVM, Random Forest) or deep learning models (e.g., BERT with fine-tuning). The model's objective should be to predict or extract keywords based on the labeled examples.

Evaluation: After training the model, evaluate its performance using appropriate metrics. You can use metrics such as precision, recall, and F1-score to assess how well the model identifies keywords in the automotive domain.

Fine-Tuning: Depending on the evaluation results, you may need to fine-tune the model to improve its keyword extraction accuracy. This can involve adjusting hyperparameters, increasing the amount of training data, or refining the labeling process.

Integration: Once you have a well-trained keyword extraction model, you can integrate it into your project. You can use this model to extract keywords from the automotive forum discussions, improving the relevance and specificity of the extracted keywords.

Continuous Learning: Keep in mind that language and terminology in the automotive field can evolve over time. It's essential to periodically update and retrain your keyword extraction model to stay current with the latest industry terminology and trends.

By training a model on a domain-specific corpus, you can leverage the specialized knowledge contained in auto repair guides to enhance the accuracy and context-awareness of your keyword extraction process. This approach should help you generate more relevant and meaningful keywords for automotive discussions, ultimately improving the quality of your analysis and insights.

## Model creation and training


The project will use the GPT-2 pre-trained model. This choice is due to the lower cost vs GPT-3 and higher. The model will be supplemented with very specific data that is was not included in the training of GPT-2.


Step 1: Prepare Your Specialized Corpus<br>
Collect Data: <br>Gather the specialized data you want to include. This could be a collection of documents, articles, or any text data that is relevant to your domain.<br>
Preprocess Data: <br>Clean your text data. This includes removing unnecessary characters, correcting typos, and possibly segmenting text into manageable sizes if you're dealing with very long documents.
<br>
<br>
Step 2: Set Up Your Environments<br>
Install Dependencies: <br>
Ensure you have PyTorch or TensorFlow installed, as these are the frameworks Hugging Face’s Transformers library supports. Also, install the Transformers library itself:<br>
Copy code<br>
!pip install torch transformers<br>
Choose a Model: <br>
Decide on the specific variant of GPT-2 you want to use (e.g., gpt2, gpt2-medium, gpt2-large, gpt2-xl). Larger models have more capacity but require more computational resources.
<br>
<br>
Step 3: Load the Pre-trained Model and Tokenizers<br>
Import necessary modules and load the pre-trained GPT-2 model and tokenizer:<br>
python<br>
Copy code:<br>
from transformers import GPT2LMHeadModel, GPT2Tokenizer
model_name = "gpt2"  # Or another variant
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
<br>
<br>
Step 4: Fine-Tune GPT-2 on Your Specialized Corpus<br>
Prepare Dataset: <br>
Tokenize your corpus using the GPT-2 tokenizer. This may involve dividing the text into sections that fit the model's maximum input size.
Fine-Tuning: <br>
Use an appropriate training loop to fine-tune the model on your data. This involves adjusting the model's weights based on the loss between the model's predictions and the actual outcomes in your specialized corpus.
It’s essential to use a suitable optimizer (like Adam) and carefully select your learning rate, batch size, and number of epochs based on the size and complexity of your data.<br>
Save Model: <br>
After fine-tuning, save your model so you can load it later for inference or further training.
<br>
<br>
Step 5: Generate Text<br>
With the fine-tuned model, you can now generate text that's more aligned with your specialized data. Use the model’s generate() function, providing a prompt from your domain to get started.

Step 2: Set Up Your Environment
Install Dependencies: Ensure you have PyTorch or TensorFlow installed, as these are the frameworks Hugging Face’s Transformers library supports. Also, install the Transformers library itself:

In [16]:
!pip install torch transformers

### Step 1: Prepare Your Specialized Corpus
Collect Data:
Gather the specialized data to include. This could be a collection of documents, articles, or any text data that is relevant to the domain.
Preprocess Data:
Clean the text data. This includes removing unnecessary characters, correcting typos, and possibly segmenting text into manageable sizes if you're dealing with very long documents.<br><br>
I have two corpora:
1. Scraped from an online forum for owner of a specific classic car. '/content/drive/MyDrive/e9/nlp/df_table_3.csv'
2. A manual for that care where the text has been digitized. '/content/drive/MyDrive/Data_sets/e9_manual.txt'

### Step 2: Set Up Your Environments
Install Dependencies:
Ensure you have PyTorch or TensorFlow installed, as these are the frameworks Hugging Face’s Transformers library supports. Also, install the Transformers library itself:
Copy code
!pip install torch transformers
Choose a Model:
Decide on the specific variant of GPT-2 you want to use (e.g., gpt2, gpt2-medium, gpt2-large, gpt2-xl). Larger models have more capacity but require more computational resources.

### Step 4: Fine-Tune GPT-2 on Your Specialized Corpus<br>
Prepare Dataset: <br>
Tokenize your corpus using the GPT-2 tokenizer. This may involve dividing the text into sections that fit the model's maximum input size.
Fine-Tuning: <br>
Use an appropriate training loop to fine-tune the model on your data. This involves adjusting the model's weights based on the loss between the model's predictions and the actual outcomes in your specialized corpus.
It’s essential to use a suitable optimizer (like Adam) and carefully select your learning rate, batch size, and number of epochs based on the size and complexity of your data.<br>
Save Model: <br>
After fine-tuning, save your model so you can load it later for inference or further training.

## Step 5: Generate Text<br>
With the fine-tuned model, you can now generate text that's more aligned with your specialized data. Use the model’s generate() function, providing a prompt from your domain to get started.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import Dataset, DataLoader
from transformers import AdamW
import torch
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Initialize tokenizer and model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# Assume lines variable is already populated from your dataset
input_ids = []
attention_masks = []
for line in lines:
    tokens = tokenizer(line, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
    input_ids.append(tokens['input_ids'])
    attention_masks.append(tokens['attention_mask'])

# Convert lists to tensors
input_ids = torch.cat(input_ids)
attention_masks = torch.cat(attention_masks)

# Split the data into train and validation sets
train_inputs, val_inputs, train_masks, val_masks = train_test_split(input_ids, attention_masks, test_size=0.1, random_state=42)

class TextDataset(Dataset):
    def __init__(self, input_ids, masks):
        self.input_ids = input_ids
        self.masks = masks

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx], 'attention_mask': self.masks[idx]}

train_dataset = TextDataset(train_inputs, train_masks)
val_dataset = TextDataset(val_inputs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 6
training_losses = []
validation_losses = []

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = input_ids.clone()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)
    training_losses.append(avg_train_loss)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = input_ids.clone()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()
    avg_val_loss = total_val_loss / len(val_loader)
    validation_losses.append(avg_val_loss)
    print(f'Epoch {epoch}, Training Loss: {avg_train_loss}, Validation Loss: {avg_val_loss}')

# Plotting
plt.plot(range(1, epochs+1), training_losses, 'bo-', label='Training loss')
plt.plot(range(1, epochs+1), validation_losses, 'ro-', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Save the model and tokenizer
model_path = '/content/drive/MyDrive/e9/pytorch'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0, Training Loss: 2.0778834871119924, Validation Loss: 1.555222636596723
Epoch 1, Training Loss: 1.7374151972432932, Validation Loss: 1.5395240069112994
Epoch 2, Training Loss: 1.6239217813644145, Validation Loss: 1.5502379357137464
Epoch 3, Training Loss: 1.5251273524430062, Validation Loss: 1.562598560682752


In [17]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import Dataset, DataLoader
from transformers import AdamW
import torch
from sklearn.model_selection import train_test_split

# Initialize tokenizer and model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)

# Load and prepare dataset
file_path = '/content/drive/MyDrive/e9/nlp/df_table_3.csv'
#file_path = '/content/drive/MyDrive/Data_sets/e9_manual.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()  # Assume each line is a separate data entry

# Tokenize each line separately to treat each as a sample
input_ids = []
attention_masks = []
for line in lines:
    tokens = tokenizer(line, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
    input_ids.append(tokens['input_ids'])
    attention_masks.append(tokens['attention_mask'])

# Convert lists to tensors
input_ids = torch.cat(input_ids)
attention_masks = torch.cat(attention_masks)

# Split the data into train and validation sets
train_inputs, val_inputs, train_masks, val_masks = train_test_split(input_ids, attention_masks, test_size=0.1, random_state=42)

class TextDataset(Dataset):
    def __init__(self, input_ids, masks):
        self.input_ids = input_ids
        self.masks = masks

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.masks[idx]
        }

# Create DataLoader for train and validation datasets
train_dataset = TextDataset(train_inputs, train_masks)
val_dataset = TextDataset(val_inputs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)  # Reduced batch size
val_loader = DataLoader(val_dataset, batch_size=1)

# Prepare for training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

# Define the training and evaluation loop
epochs = 4

for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = input_ids.clone()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation phase
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = input_ids.clone()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print(f'Epoch {epoch}, Validation Loss: {avg_val_loss}')

# Save the fine-tuned model and tokenizer
model_path = '/content/drive/MyDrive/e9/pytorch'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0, Validation Loss: 1.586045045744289
Epoch 1, Validation Loss: 1.5434075289151885
Epoch 2, Validation Loss: 1.5484958629716525
Epoch 3, Validation Loss: 1.5630228106271138


('/content/drive/MyDrive/e9/pytorch/tokenizer_config.json',
 '/content/drive/MyDrive/e9/pytorch/special_tokens_map.json',
 '/content/drive/MyDrive/e9/pytorch/vocab.json',
 '/content/drive/MyDrive/e9/pytorch/merges.txt',
 '/content/drive/MyDrive/e9/pytorch/added_tokens.json')

Step 3: Load the Pre-trained Model and Tokenizer
Import necessary modules and load the pre-trained GPT-2 model and tokenizer:

## Parking Lot

## Load tables into Snowflake

In [18]:
# Old code on determining post vs thread



        # Parse the HTML content of the page
#        soup = BeautifulSoup(page.content, "html.parser")
#        results = soup.find_all('div', class_='contentRow-main')

#        for result in results:
#            title_element = result.find('h3', class_='contentRow-title')
#            if title_element and title_element.find('a'):
#                title = title_element.get_text(strip=True)
#                url = title_element.find('a')['href']

                # Check if it's a thread or a post
#                post_info = result.find('div', class_='contentRow-minor').get_text(strip=True)
#                if "Thread" in post_info:
#                    type = "Thread"

 #                   print(f"Title: {title}, URL: {url}, Type: {type}")
 #                   print('--------------------------------------------------')
                # Append the information to a list
#                    url = 'https://e9coupe.com'+url
#                    thread_urls.append({'url': url})
#                    #thread_urls.append({'title': title, 'url': url, 'type': type})


# Create a DataFrame for the URLs
#df_threads = pd.DataFrame(thread_urls)